# Ticket Priority Classifier

End-to-end ML pipeline: feature engineering → model training → evaluation → registry deployment.

**Goal:** Predict support ticket priority (Critical, High, Medium, Low) from ticket attributes, customer history, and product info.

**Models:** RandomForestClassifier vs LogisticRegression

**Prerequisites:** Phases 1-3 complete (setup, data creation, processing).

## 1. Setup

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T

from snowflake.ml.modeling.preprocessing import OrdinalEncoder, StandardScaler
from snowflake.ml.modeling.ensemble import RandomForestClassifier
from snowflake.ml.modeling.linear_model import LogisticRegression
from snowflake.ml.modeling.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from snowflake.ml.registry import Registry

session = get_active_session()
session.use_role("DEMO_ML_ENGINEER")
session.use_warehouse("DEMO_ML_WH")
session.use_database("MSFT_SNOWFLAKE_DEMO")
session.use_schema("ML")

print(session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_DATABASE()").collect())

## 2. Feature Engineering

In [ ]:
# Load source tables
tickets_df = session.table("RAW.SUPPORT_TICKETS")
customers_df = session.table("RAW.CUSTOMERS")
products_df = session.table("RAW.PRODUCTS")
orders_df = session.table("RAW.ORDERS")

# Customer order history
customer_history = (
    orders_df
    .group_by("CUSTOMER_ID")
    .agg(
        F.count("ORDER_ID").alias("CUSTOMER_ORDER_COUNT"),
        F.sum("TOTAL_AMOUNT").alias("CUSTOMER_TOTAL_SPEND"),
        F.avg("TOTAL_AMOUNT").alias("CUSTOMER_AVG_ORDER"),
        F.max("ORDER_DATE").alias("CUSTOMER_LAST_ORDER"),
    )
)

# Build feature set
features_raw = (
    tickets_df
    .join(customers_df.select(
        "CUSTOMER_ID", "CUSTOMER_SEGMENT", "STATE", "REGISTRATION_DATE"
    ), on="CUSTOMER_ID", how="left")
    .join(products_df.select(
        "PRODUCT_ID", "CATEGORY", "UNIT_PRICE"
    ), on="PRODUCT_ID", how="left")
    .join(customer_history, on="CUSTOMER_ID", how="left")
    .with_column("TICKET_DESC_LENGTH", F.length(F.col("TICKET_DESCRIPTION")))
    .with_column("TICKET_SUBJECT_LENGTH", F.length(F.col("TICKET_SUBJECT")))
    .with_column("TICKET_WORD_COUNT",
        F.array_size(F.split(F.col("TICKET_DESCRIPTION"), F.lit(" "))))
    .with_column("DAYS_AS_CUSTOMER",
        F.datediff("day", F.col("REGISTRATION_DATE"), F.col("CREATED_AT")))
    .with_column("TICKET_HOUR", F.hour(F.col("CREATED_AT")))
    .with_column("TICKET_DAY_OF_WEEK", F.dayofweek(F.col("CREATED_AT")))
    .with_column("HAS_PRODUCT",
        F.when(F.col("PRODUCT_ID").is_not_null(), 1).otherwise(0))
    .na.fill({
        "CUSTOMER_ORDER_COUNT": 0,
        "CUSTOMER_TOTAL_SPEND": 0.0,
        "CUSTOMER_AVG_ORDER": 0.0,
        "UNIT_PRICE": 0.0,
        "DAYS_AS_CUSTOMER": 0,
    })
    .select(
        "TICKET_ID", "PRIORITY",
        "CATEGORY", "CUSTOMER_SEGMENT", "STATUS",
        "TICKET_DESC_LENGTH", "TICKET_SUBJECT_LENGTH", "TICKET_WORD_COUNT",
        "DAYS_AS_CUSTOMER", "TICKET_HOUR", "TICKET_DAY_OF_WEEK",
        "HAS_PRODUCT", "UNIT_PRICE",
        "CUSTOMER_ORDER_COUNT", "CUSTOMER_TOTAL_SPEND", "CUSTOMER_AVG_ORDER",
    )
)

print(f"Raw features: {features_raw.count():,} rows")
features_raw.show(5)

In [ ]:
# Encode target: PRIORITY → ordinal
priority_encoder = OrdinalEncoder(
    input_cols=["PRIORITY"],
    output_cols=["PRIORITY_ENCODED"],
    categories={"PRIORITY": ["Low", "Medium", "High", "Critical"]},
)
features = priority_encoder.fit(features_raw).transform(features_raw)

# Encode categoricals
cat_encoder = OrdinalEncoder(
    input_cols=["CATEGORY", "CUSTOMER_SEGMENT", "STATUS"],
    output_cols=["CATEGORY_ENCODED", "SEGMENT_ENCODED", "STATUS_ENCODED"],
)
features = cat_encoder.fit(features).transform(features)

# Scale numerics
numeric_cols = [
    "TICKET_DESC_LENGTH", "TICKET_SUBJECT_LENGTH", "TICKET_WORD_COUNT",
    "DAYS_AS_CUSTOMER", "TICKET_HOUR", "TICKET_DAY_OF_WEEK",
    "UNIT_PRICE", "CUSTOMER_ORDER_COUNT", "CUSTOMER_TOTAL_SPEND",
    "CUSTOMER_AVG_ORDER",
]
scaled_cols = [f"{c}_SCALED" for c in numeric_cols]

scaler = StandardScaler(input_cols=numeric_cols, output_cols=scaled_cols)
features = scaler.fit(features).transform(features)

# Final feature table
final_cols = (
    ["TICKET_ID", "PRIORITY", "PRIORITY_ENCODED"]
    + ["CATEGORY_ENCODED", "SEGMENT_ENCODED", "STATUS_ENCODED", "HAS_PRODUCT"]
    + scaled_cols
)
features = features.select(final_cols)
features.write.save_as_table("ML.TICKET_CLASSIFICATION_FEATURES", mode="overwrite")

print(f"Saved ML.TICKET_CLASSIFICATION_FEATURES — {features.count():,} rows, {len(features.columns)} cols")

## 3. Explore Data

In [ ]:
features_df = session.table("ML.TICKET_CLASSIFICATION_FEATURES")

FEATURE_COLS = [
    "CATEGORY_ENCODED", "SEGMENT_ENCODED", "STATUS_ENCODED", "HAS_PRODUCT",
    "TICKET_DESC_LENGTH_SCALED", "TICKET_SUBJECT_LENGTH_SCALED",
    "TICKET_WORD_COUNT_SCALED", "DAYS_AS_CUSTOMER_SCALED",
    "TICKET_HOUR_SCALED", "TICKET_DAY_OF_WEEK_SCALED",
    "UNIT_PRICE_SCALED", "CUSTOMER_ORDER_COUNT_SCALED",
    "CUSTOMER_TOTAL_SPEND_SCALED", "CUSTOMER_AVG_ORDER_SCALED",
]
TARGET_COL = "PRIORITY_ENCODED"
LABEL_COL = "PRIORITY"

print("Class distribution:")
features_df.group_by(LABEL_COL).agg(F.count("*").alias("COUNT")).sort(LABEL_COL).show()

features_df.describe().show()

## 4. Train / Test Split

In [ ]:
train_df, test_df = features_df.random_split([0.8, 0.2], seed=42)

print(f"Training: {train_df.count():,} rows")
print(f"Test:     {test_df.count():,} rows")

print("\nTraining class distribution:")
train_df.group_by(LABEL_COL).agg(F.count("*").alias("COUNT")).sort(LABEL_COL).show()

## 5. Model 1 — Random Forest

In [ ]:
rf_model = RandomForestClassifier(
    input_cols=FEATURE_COLS,
    label_cols=[TARGET_COL],
    output_cols=["RF_PREDICTED"],
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(train_df)

rf_preds = rf_model.predict(test_df)

rf_accuracy = accuracy_score(df=rf_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["RF_PREDICTED"])
rf_precision = precision_score(df=rf_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["RF_PREDICTED"], average="weighted")
rf_recall = recall_score(df=rf_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["RF_PREDICTED"], average="weighted")
rf_f1 = f1_score(df=rf_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["RF_PREDICTED"], average="weighted")

print(f"RandomForest Results:")
print(f"  Accuracy:  {rf_accuracy:.4f}")
print(f"  Precision: {rf_precision:.4f}")
print(f"  Recall:    {rf_recall:.4f}")
print(f"  F1-Score:  {rf_f1:.4f}")

rf_cm = confusion_matrix(df=rf_preds, y_true_col_name=TARGET_COL, y_pred_col_name="RF_PREDICTED")
print(f"\nConfusion Matrix:\n{rf_cm}")

## 6. Model 2 — Logistic Regression

In [ ]:
lr_model = LogisticRegression(
    input_cols=FEATURE_COLS,
    label_cols=[TARGET_COL],
    output_cols=["LR_PREDICTED"],
    max_iter=1000,
    multi_class="multinomial",
    solver="lbfgs",
    random_state=42,
)
lr_model.fit(train_df)

lr_preds = lr_model.predict(test_df)

lr_accuracy = accuracy_score(df=lr_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["LR_PREDICTED"])
lr_precision = precision_score(df=lr_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["LR_PREDICTED"], average="weighted")
lr_recall = recall_score(df=lr_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["LR_PREDICTED"], average="weighted")
lr_f1 = f1_score(df=lr_preds, y_true_col_names=[TARGET_COL], y_pred_col_names=["LR_PREDICTED"], average="weighted")

print(f"LogisticRegression Results:")
print(f"  Accuracy:  {lr_accuracy:.4f}")
print(f"  Precision: {lr_precision:.4f}")
print(f"  Recall:    {lr_recall:.4f}")
print(f"  F1-Score:  {lr_f1:.4f}")

lr_cm = confusion_matrix(df=lr_preds, y_true_col_name=TARGET_COL, y_pred_col_name="LR_PREDICTED")
print(f"\nConfusion Matrix:\n{lr_cm}")

## 7. Compare Models

In [ ]:
print(f"{'Metric':<12} {'RandomForest':>14} {'LogisticReg':>14}")
print("-" * 42)
print(f"{'Accuracy':<12} {rf_accuracy:>14.4f} {lr_accuracy:>14.4f}")
print(f"{'Precision':<12} {rf_precision:>14.4f} {lr_precision:>14.4f}")
print(f"{'Recall':<12} {rf_recall:>14.4f} {lr_recall:>14.4f}")
print(f"{'F1-Score':<12} {rf_f1:>14.4f} {lr_f1:>14.4f}")

best_name = "RandomForest" if rf_f1 >= lr_f1 else "LogisticRegression"
best_model = rf_model if rf_f1 >= lr_f1 else lr_model
best_f1 = max(rf_f1, lr_f1)
print(f"\n\u2192 Best model: {best_name} (F1={best_f1:.4f})")

## 8. Save Predictions & Feature Importance

In [ ]:
# Save RF predictions
rf_output = (
    rf_preds
    .with_column("MODEL_NAME", F.lit("RandomForestClassifier"))
    .with_column("MODEL_VERSION", F.lit("v1"))
    .select("TICKET_ID", "PRIORITY", "PRIORITY_ENCODED",
            "RF_PREDICTED", "MODEL_NAME", "MODEL_VERSION")
    .with_column_renamed("RF_PREDICTED", "PREDICTED_PRIORITY")
)
rf_output.write.save_as_table("ML.CLASSIFICATION_PREDICTIONS", mode="overwrite")
print(f"Saved ML.CLASSIFICATION_PREDICTIONS — {rf_output.count():,} rows")

# Feature importance
try:
    sklearn_model = rf_model.to_sklearn()
    importances = sklearn_model.feature_importances_
    print("\nFeature Importance (RandomForest):")
    for feat, imp in sorted(zip(FEATURE_COLS, importances), key=lambda x: x[1], reverse=True):
        bar = "\u2588" * int(imp * 50)
        print(f"  {feat:<40} {imp:.4f} {bar}")
except Exception as e:
    print(f"Could not extract feature importance: {e}")

## 9. Register Model

In [ ]:
registry = Registry(session=session, database_name="MSFT_SNOWFLAKE_DEMO", schema_name="ML")

model_version = registry.log_model(
    model=rf_model,
    model_name="TICKET_PRIORITY_CLASSIFIER",
    version_name="v1",
    comment="RandomForestClassifier for support ticket priority prediction. "
            "Trained on ticket attributes, customer history, and product info.",
    metrics={
        "accuracy": rf_accuracy,
        "f1_weighted": rf_f1,
        "precision_weighted": rf_precision,
        "recall_weighted": rf_recall,
        "training_rows": train_df.count(),
        "test_rows": test_df.count(),
        "n_features": len(FEATURE_COLS),
        "algorithm": "RandomForestClassifier",
        "n_estimators": 100,
        "max_depth": 10,
    },
    sample_input_data=train_df.select(FEATURE_COLS).limit(10),
)

# Set as default
registry.get_model("TICKET_PRIORITY_CLASSIFIER").default = "v1"

print(f"Registered: TICKET_PRIORITY_CLASSIFIER v1")
print(f"Metrics: {model_version.show_metrics()}")

## 10. Inference from Registry

In [ ]:
loaded_model = registry.get_model("TICKET_PRIORITY_CLASSIFIER").version("v1")

sample = test_df.select(FEATURE_COLS).limit(5)
results = loaded_model.run(sample, function_name="predict")
results.show()

## SQL Inference

After registration, call the model directly from SQL:

```sql
SELECT
    t.TICKET_ID,
    t.PRIORITY AS ACTUAL_PRIORITY,
    ML.TICKET_PRIORITY_CLASSIFIER!PREDICT(
        t.CATEGORY_ENCODED, t.SEGMENT_ENCODED, t.STATUS_ENCODED,
        t.HAS_PRODUCT, t.TICKET_DESC_LENGTH_SCALED,
        t.TICKET_SUBJECT_LENGTH_SCALED, t.TICKET_WORD_COUNT_SCALED,
        t.DAYS_AS_CUSTOMER_SCALED, t.TICKET_HOUR_SCALED,
        t.TICKET_DAY_OF_WEEK_SCALED, t.UNIT_PRICE_SCALED,
        t.CUSTOMER_ORDER_COUNT_SCALED, t.CUSTOMER_TOTAL_SPEND_SCALED,
        t.CUSTOMER_AVG_ORDER_SCALED
    ) AS PREDICTED_PRIORITY
FROM ML.TICKET_CLASSIFICATION_FEATURES t
LIMIT 10;
```